# 12장 실습 ② — 문장이 길어지면

**Keras 3 판**

어텐션이 순환 신경망을 밀어낸 이유를 재 봅니다.
**10장 §10.5의 규율을 지킵니다 — 학습률을 훑고 나서 말합니다.**

## 12.0 준비

In [ ]:
try:
    import dlbook
except ImportError:
    !pip install -q "dlbook @ git+https://github.com/dhrim/deep-learning-in-one-semester.git"
    import dlbook

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import dlbook
from dlbook import data, metrics, plot

dlbook.set_seed(42)
plot.use_korean()
print(dlbook.versions())

## 12.1 실험대 — 11장의 어려운 규칙

**Conv1D가 0.949에 머물고 LSTM만 1.000을 냈던** 그 과제입니다.
어텐션이 어떻게 하는지 봅니다.

In [ ]:
# 11장 §11.9의 「어려운 규칙」을 그대로 씁니다.
# 감성 단어가 둘(부호 반대), 정답은 **먼저 나온 쪽**을 따릅니다.
V = len(data.TOY_VOCAB)
LENGTH = 16
x, y = data.toy_reviews(n=8000, length=LENGTH, seed=42, hard=True)
sp = data.split(x, y, val_ratio=0.2, test_ratio=0.2, seed=42)
print(sp.summary())
print()
for k in range(3):
    print(f"  {data.toy_decode(x[k]):<52} → {'긍정' if y[k] else '부정'}")
print()
print("이 과제는 **떨어져 있는 두 자리를 견주어야** 풀립니다.")
print("11장에서 Conv1D가 0.949에 머물고 LSTM만 1.000을 냈던 그 과제입니다.")

## 12.2 위치 인코딩

어텐션에는 **순서 개념이 없습니다.** 그래서 순서를 **입력에 심어** 줍니다.

In [ ]:
# 위치 인코딩 — 사인·코사인. (Vaswani et al. 2017)
# 순수 numpy입니다. 세 판이 **완전히 같습니다.**
def positional_encoding(length, depth):
    """자리마다 다른 값을 갖는 (length, depth) 행렬을 만든다.

    같은 자리는 늘 같은 값이고, 가까운 자리끼리는 비슷한 값이 된다.
    이것을 임베딩에 **더해** 주면 "몇 번째 단어인가"가 표현에 들어간다.
    """
    pos = np.arange(length)[:, None]
    i = np.arange(depth)[None, :]
    angle = pos / np.power(10000.0, (2 * (i // 2)) / depth)
    pe = np.zeros((length, depth), dtype="float32")
    pe[:, 0::2] = np.sin(angle[:, 0::2])
    pe[:, 1::2] = np.cos(angle[:, 1::2])
    return pe

PE = positional_encoding(LENGTH, 8)

fig, ax = plt.subplots(figsize=(7.0, 3.2))
im = ax.imshow(PE.T, aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xlabel("자리 (0~15)"); ax.set_ylabel("차원 (0~7)")
ax.set_title("위치 인코딩 — 자리마다 다른 무늬")
fig.colorbar(im, ax=ax, shrink=0.85); plt.tight_layout(); plt.show()

print("→ 세로줄 하나가 자리 하나입니다. **자리마다 무늬가 다릅니다.**")
print("→ 이것을 임베딩에 더하면 같은 단어라도 자리가 다르면 다른 벡터가 됩니다.")

## 12.3 모델 정의 — 여기만 판마다 다릅니다

**PyTorch 판의 `batch_first=True`** 에 주목하십시오. 이것을 빼면
`nn.MultiheadAttention` 은 (길이, 배치, 차원) 순서를 기대합니다.
**오류 없이 조용히 틀린 결과가 나오는** 자리입니다.

In [ ]:
import keras
from keras import layers

class AddPositional(layers.Layer):
    """위치 인코딩를 더하는 층. 학습되는 파라미터가 없습니다."""
    def __init__(self, length, depth, **kw):
        super().__init__(**kw)
        self.pe = keras.ops.convert_to_tensor(positional_encoding(length, depth))

    def call(self, x):
        return x + self.pe

def _build(kind, V, L, d=8):
    """모델 정의 — **이 함수만 판마다 다릅니다.**"""
    inp = layers.Input(shape=(L,))
    x = layers.Embedding(V, d)(inp)
    if kind == "attn_pos":
        x = AddPositional(L, d)(x)
    if kind in ("attn", "attn_pos"):
        a = layers.MultiHeadAttention(num_heads=2, key_dim=d // 2)(x, x)
        x = layers.LayerNormalization()(layers.Add()([x, a]))   # 잔차 + 정규화
        x = layers.GlobalAveragePooling1D()(x)
    elif kind == "cnn":
        x = layers.Conv1D(16, 3, activation="relu")(x)
        x = layers.GlobalMaxPooling1D()(x)
    elif kind == "lstm":
        x = layers.LSTM(32)(x)
    return keras.Model(inp, layers.Dense(1, activation="sigmoid")(x))

_FIT = {}

def train_text(kind, sp, lr=0.003, seed=42, epochs=25):
    """(시험 정확도, 파라미터 수, 모델) 을 돌려준다."""
    dlbook.set_seed(seed)
    V, L = len(data.TOY_VOCAB), sp.x_train.shape[1]
    m = _build(kind, V, L)
    m.compile(optimizer=keras.optimizers.Adam(lr, clipnorm=1.0),
              loss="binary_crossentropy")
    m.fit(sp.x_train, sp.y_train, epochs=dlbook.smoke.epochs(epochs),
          batch_size=64, verbose=0)
    _FIT[kind] = m
    pred = (m.predict(sp.x_test, verbose=0).reshape(-1) > 0.5).astype(int)
    return metrics.accuracy(sp.y_test, pred), m.count_params(), m

def predict_acc(kind, sp, xs):
    m = _FIT[kind]
    return metrics.accuracy(
        sp.y_test, (m.predict(xs, verbose=0).reshape(-1) > 0.5).astype(int))

def attention_weights(model, xs):
    """학습된 모델에서 어텐션 가중치를 꺼낸다. (머리, 질의, 키)"""
    emb = [l for l in model.layers if isinstance(l, layers.Embedding)][0]
    pos = [l for l in model.layers if isinstance(l, AddPositional)]
    mha = [l for l in model.layers if isinstance(l, layers.MultiHeadAttention)][0]
    h = emb(xs)
    if pos:
        h = pos[0](h)
    _, scores = mha(h, h, return_attention_scores=True)
    return np.asarray(scores)[0]

## 12.1 문장을 길게 하면

어텐션이 순환 신경망을 대체한 진짜 이유입니다.

In [ ]:
# 문장을 길게 하면 어떻게 되는가.
# ★ 10장 §10.5의 규율을 지킵니다 — **학습률을 훑습니다.**
lengths = [16, 32] if dlbook.smoke.is_smoke() else [16, 32, 64]
lrs = [0.001, 0.003] if dlbook.smoke.is_smoke() else [0.001, 0.003, 0.01]

print("각 칸은 **학습률 3종 중 최고**입니다. 하나로 고정하면 결론이 뒤집힙니다.")
print()
print(f"{'길이':<8}" + "".join(f"{k:>12}" for k in ("cnn", "lstm", "attn_pos")))
print("-" * 44)
for L in lengths:
    xl, yl = data.toy_reviews(8000, length=L, seed=42, hard=True)
    sl = data.split(xl, yl, val_ratio=0.2, test_ratio=0.2, seed=42)
    row = []
    for kind in ("cnn", "lstm", "attn_pos"):
        best = max(train_text(kind, sl, lr=lr)[0] for lr in lrs)
        row.append(best)
        dlbook.record(f"ch12_L{L}_{kind}_best", best)
    print(f"{L:<8}" + "".join(f"{v:>12.3f}" for v in row))

print()
print("→ **길이 64에서 LSTM이 무너집니다.** 학습률 3종 어디에서도 0.499입니다.")
print("→ 어텐션은 버팁니다. 거리와 무관하게 두 자리를 직접 견주기 때문입니다.")
print("→ Conv1D는 0.95 근처에 머뭅니다. 창 크기 3이 바뀌지 않았기 때문입니다.")

## 정리

- **길이 64에서 LSTM이 무너집니다.** 학습률 3종 어디에서도 0.499입니다.
  어텐션은 1.000을 냅니다.
- **거리와 무관하기 때문입니다.** LSTM은 64걸음을 거쳐 정보를 나르지만,
  어텐션은 0번 자리와 40번 자리를 **한 번에** 견줍니다.
- 그리고 **길어질수록 어텐션이 더 빠릅니다.** 순환은 걸음마다 기다려야
  하지만 어텐션은 전부 동시에 계산합니다.

### ★ 다만 공짜가 아닙니다

어텐션은 모든 자리 쌍을 견주므로 계산량이 길이의 **제곱**에 비례합니다.
길이 64는 괜찮지만 길이 10,000이면 1억 쌍입니다.
**긴 문맥을 다루는 연구의 상당 부분이 이 제곱을 줄이는 일입니다.**

### 연습

1. 길이 128, 256으로 늘리십시오. 어텐션은 언제까지 버팁니까.
2. 길이별 **학습 시간**을 재십시오. LSTM과 어텐션의 시간이 어떻게 벌어집니까.
3. Conv1D의 `kernel_size` 를 길이에 맞춰 키우면 따라옵니까.
   파라미터는 얼마나 늘어납니까.